### Before we build the API, we must compress this model.t5 so it can run on CPU servers.

In [1]:
import tensorflow as tf


In [3]:
saved_model_path = "../models/plant_model_best.keras"
model = tf.keras.models.load_model(saved_model_path)

2026-04-13 19:54:53.783832: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-04-13 19:54:53.784061: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-13 19:54:53.784067: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-13 19:54:53.784791: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-13 19:54:53.785478: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [10]:
#there is a version issue with the .from_keras_model function because the versions of tensorflow 
# and keras don't match plus the saved model way is also problematic because Dropout layer 
#in the model uses random seed generators and when we try to convert model for inference,
#tensorflow gets confused and refuses to export the seed generators 

#what we can do is wrap model is a tf.function, putting the model in a training = false mode
#this forces tensorflow to strip away dropout seed generators and retain the math

In [11]:
@tf.function(input_signature = [tf.TensorSpec(shape = [1,224,224,3], dtype = tf.float32)])
def serving_fn(inputs):
    return model(inputs, training=False)

In [12]:
#tracing graph to create Concrete function
concrete_func = serving_fn.get_concrete_function()
print('Graph Traced, seed generators stripped.')

Graph Traced, seed generators stripped.


In [13]:
#now we feed the purely math inferences in the converter
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])

In [14]:
#apply the Int8 operation
converter.optimizations = [tf.lite.Optimize.DEFAULT]

In [15]:
tflite_quant_model = converter.convert()

2026-04-13 20:32:41.379143: I tensorflow/core/grappler/devices.cc:75] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
2026-04-13 20:32:41.383390: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-04-13 20:32:41.389161: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-13 20:32:41.389211: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
W0000 00:00:1776092562.482501  540752 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1776092562.482566  540752 tf_tfl_flatbuffer_helpers.cc:393] Ignored 

In [16]:
tflite_save_path = "../models/plant_model_quantized.tflite"
with open(tflite_save_path, 'wb') as f:
    f.write(tflite_quant_model)